# TENSORIX: Colab notebook

One notebook for every run (baseline now, distillation later). Set the runtime to **T4 GPU** first (Runtime, Change runtime type).

**Every new session, in order:**
1. Run **Cell 1 (Setup)**. Colab wipes its disk between sessions, so this is always needed.
2. Edit the values at the top of **Cell 2 (Train)** and run it. After a disconnect, rerun Cell 1 then Cell 2 unchanged: training resumes from the last save.
3. Run **Cell 3** for the log table and curves, and **Cell 4** for sample text.

Do not use "Run all". Code and data live in git and on Drive, never only on Colab.

In [ ]:
# CELL 1: SETUP (run first in every new session)
import os, torch

BRANCH = "v2-rebuild"      # change to "main" once the v2 PR is merged
REPO   = "https://github.com/sunidhishekar-create/tensorix-kd-slm.git"
DRIVE  = "/content/drive/MyDrive/tensorix/v2"

from google.colab import drive
drive.mount("/content/drive")

%cd /content
if not os.path.exists("/content/tensorix-kd-slm"):
    !git clone {REPO}
%cd /content/tensorix-kd-slm
!git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
!pip install -q -r requirements.txt

# data goes on Colab's local disk (reading from Drive during training is slow)
os.makedirs("/content/data", exist_ok=True)
if not os.path.exists("/content/data/train.bin"):
    !cp {DRIVE}/data/* /content/data/
!ls -lh /content/data

need = ["train.bin", "val_c4.bin", "val_ts.bin", "meta.json"]
missing = [f for f in need if not os.path.exists(f"/content/data/{f}")]
assert not missing, f"missing in /content/data: {missing}. Check that {DRIVE}/data has them."
assert torch.cuda.is_available(), "No GPU. Runtime, Change runtime type, T4 GPU, then rerun Cell 1."
print("GPU:", torch.cuda.get_device_name(0), "| setup OK")

In [ ]:
# CELL 2: TRAIN. Edit these three values, then run.
RUN_NAME    = "baseline_full"   # NEW name for every new run; reusing a name RESUMES that run
MAX_STEPS   = 20000             # SHORT tier = 5000, FULL tier = 20000. Speed test: 100 with RUN_NAME "speedtest"
EXTRA_FLAGS = ""                # e.g. "--grad_accum 4" if CUDA runs out of memory; KD flags go here later

# Do NOT change batch size, block size or --eval_windows between runs: the comparison depends on them.
SHORT = MAX_STEPS <= 5000
EVAL_EVERY, SAVE_EVERY = (250, 500) if SHORT else (1000, 1000)
OUT = f"/content/{RUN_NAME}" if RUN_NAME == "speedtest" else f"{DRIVE}/checkpoints/{RUN_NAME}"

if os.path.exists(f"{OUT}/ckpt.pt"):
    print(f"Found a checkpoint in {OUT}: this run will RESUME. Use a new RUN_NAME for a fresh run.")

%cd /content/tensorix-kd-slm
!python src/train.py --data_dir /content/data --out_dir {OUT} --max_steps {MAX_STEPS} --eval_interval {EVAL_EVERY} --save_interval {SAVE_EVERY} {EXTRA_FLAGS}

In [ ]:
# CELL 3: RESULTS. Log table and validation curves. Add run names to compare runs.
import pandas as pd, matplotlib.pyplot as plt
RUNS = ["baseline_short", "baseline_full"]    # later: add the KD runs here

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for r in RUNS:
    p = f"{DRIVE}/checkpoints/{r}/log.csv"
    if not os.path.exists(p):
        print("missing:", p); continue
    df = pd.read_csv(p)
    print(r); print(df.tail(3).to_string(index=False)); print()
    ax[0].plot(df.step, df.ppl_c4, marker="o", label=r)
    ax[1].plot(df.step, df.ppl_ts, marker="o", label=r)
for a, name in zip(ax, ["C4", "TinyStories"]):
    a.set(title=f"Validation perplexity: {name}", xlabel="step", ylabel="perplexity", yscale="log")
    a.grid(alpha=0.3); a.legend()
os.makedirs(f"{DRIVE}/results", exist_ok=True)
plt.savefig(f"{DRIVE}/results/val_ppl.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# CELL 4: SAMPLES. Generate text from a trained checkpoint (sanity check, and material for the demo).
import sys
sys.path.insert(0, "/content/tensorix-kd-slm/src")
from model import load_checkpoint
from transformers import AutoTokenizer

SAMPLE_RUN = "baseline_short"
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")
model, ck = load_checkpoint(f"{DRIVE}/checkpoints/{SAMPLE_RUN}/best.pt", device="cuda")
print(f"{SAMPLE_RUN}: best checkpoint from step {ck['step']}\n")
for prompt in ["Once upon a time, a little robot", "The city council announced on Monday that"]:
    ids = torch.tensor([tok(prompt)["input_ids"]], device="cuda")
    out = model.generate(ids, max_new_tokens=80)
    print(tok.decode(out[0].tolist()), "\n" + "-" * 60)

## Notes

- **Drive layout:** `tensorix/v2/data/` (train.bin, val_c4.bin, val_ts.bin, meta.json), `tensorix/v2/checkpoints/<run>/` (ckpt.pt, best.pt, log.csv), `tensorix/v2/results/`.
- **Budget tiers:** SHORT = 5,000 steps, FULL = 20,000 steps, both batch 64, block 256. Each tier has its own matched baseline.
- **Timing (T4, measured):** baseline about 535 ms/step, so SHORT is about 45 min and FULL about 3 h. Distillation runs are roughly 3.3x that (estimate).
- **Out of memory:** set `EXTRA_FLAGS = "--grad_accum 4"` in Cell 2. The effective batch stays 64.